# Word prediction using RNN

1.  **Input**: A 'word' (Deep')
2.  **Model**: A Recurrent Neural Network (RNN) that remembers context.
3.  **Output**: The next word (e.g., 'learning ')

By training on a definition of deep learning, the model will learn the *style*, *words*, and *structure* of the text!

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
torch.manual_seed(42)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Python312\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "c:\Python312\Lib\site-packages\tornado\platform\asyncio.py", line 211, in 

### 1. The Data
Deep Learning Definitions

In [2]:
# Create Mappings
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

def string_to_tensor(seq):
    indices = [char_to_ix[ch] for ch in seq]
    return torch.tensor(indices, dtype=torch.long)

print(f"'Deep' -> {string_to_tensor('Deep')}")

NameError: name 'chars' is not defined

In [ ]:
sentence = """
Deep learning is a type of machine learning (ML) and artificial intelligence (AI) that uses multi-layered artificial neural networks, 
modeled after the human brain, to learn complex patterns from massive amounts of data (like text, images, sounds) with minimal 
human intervention, enabling advanced capabilities in tasks such as speech recognition, image recognition, and natural language processing (NLP).
The "deep" in deep learning refers to the many layers of these neural networks, allowing them to process information hierarchically, 
from simple features to highly complex concepts, and continuously improve performance as they get more data
"""

print(f"Length of sentence: {len(sentence)} characters")

### 2. Preprocessing
Computers understand specific numbers, not characters. We need to create a **Dictionary** mapping.

In [ ]:
# Get unique characters
chars = sorted(list(set(sentence)))
print(f"Unique characters: {len(chars)}")
print(chars)

In [ ]:
# Create character to index and index to character mappings
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

def string_to_tensor(string):
    # Convert a string to a tensor of character indices
    tensor = torch.zeros(len(string)).long()
    for i, char in enumerate(string):
        tensor[i] = char_to_ix[char]
    return tensor

def tensor_to_string(tensor):
    # Convert a tensor of character indices back to a string
    string = ""
    for i in tensor:
        string += ix_to_char[i.item()]
    return string

### 3. The Dataset
We train the model to predict the NEXT character.
*   **Input**: "Two roads"
*   **Target**: "wo roads "

Notice the target is just the input shifted by one!

In [ ]:
def get_training_batch(chunk_len=200):
    # Randomly pick a chunk of text
    start_idx = np.random.randint(0, len(sentence) - chunk_len)
    end_idx = start_idx + chunk_len + 1
    text_chunk = sentence[start_idx:end_idx]
    
    input_seq = string_to_tensor(text_chunk[:-1])
    target_seq = string_to_tensor(text_chunk[1:])
    
    # Add Batch Dimension [Batch, Seq]
    return input_seq.unsqueeze(0), target_seq.unsqueeze(0)

### 4. The Model (Char-RNN)
We use an **Embedding Layer** to represent characters, an **RNN** to process sequence, and a **Linear Layer** to predict score for next char.

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, n_layers=1):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        
        # Embedding: Maps character index to a dense vector
        self.embedding = nn.Embedding(input_size, hidden_size)
        
        # RNN Layer
        self.rnn = nn.RNN(hidden_size, hidden_size, n_layers, batch_first=True)
        
        # Output layer
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, hidden):
        # x shape: [Batch, Seq]
        embedded = self.embedding(x)
        
        # RNN Output
        # out shape: [Batch, Seq, Hidden]
        out, hidden = self.rnn(embedded, hidden)
        
        # Reshape for Linear Layer to [Batch*Seq, Hidden]
        out = out.reshape(-1, self.hidden_size)
        
        # Prediction
        out = self.fc(out)
        return out, hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(self.n_layers, batch_size, self.hidden_size)

# Hyperparameters
n_chars = len(chars)
hidden_size = 128
n_layers = 1

model = CharRNN(n_chars, hidden_size, n_chars, n_layers)

### 5. Generation Function
Before training, let's see what the untrained model writes.

In [ ]:
def generate(model, start_str="Two", predict_len=100, temperature=0.8):
    hidden = model.init_hidden(1)
    input_seq = string_to_tensor(start_str).unsqueeze(0)
    
    predicted = start_str
    
    # "Warm up" the hidden state with start string
    # We iterate through the start string except the last char
    # because we want to predict FROM the last char.
    
    for i in range(len(start_str) - 1):
        _, hidden = model(input_seq[:, i].unsqueeze(1), hidden)
    
    # Last char to start prediction
    inp = input_seq[:, -1].unsqueeze(1)
    
    for i in range(predict_len):
        output, hidden = model(inp, hidden)
        
        # Apply temperature to sample more creatively
        output_dist = output.data.view(-1).div(temperature).exp()
        top_i = torch.multinomial(output_dist, 1)[0]
        
        # Add predicted char to string
        char = ix_to_char[top_i.item()]
        predicted += char
        
        # Use predicted char as next input
        inp = string_to_tensor(char).unsqueeze(0)
        
    return predicted

# Try generation (Untrained)
print("Untrained Randomness:")
print(generate(model))

### 6. Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

n_epochs = 2000
print_every = 200
loss_history = []

for epoch in range(1, n_epochs + 1):
    input_seq, target_seq = get_training_batch(chunk_len=100)
    hidden = model.init_hidden(1)
    
    model.zero_grad()
    
    output, hidden = model(input_seq, hidden)
    
    # target_seq needs to be flattened for CrossEntropy
    loss = criterion(output, target_seq.view(-1))
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())
    
    if epoch % print_every == 0:
        print(f'Epoch: {epoch} | Loss: {loss.item():.4f}')
        print("Sample Generation:")
        print(generate(model, start_str="The", predict_len=50))

### 7. Results Visualization

In [ ]:
plt.plot(loss_history)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
print("FINAL GENERATED sentence:\n")

print(generate(model, start_str="Two roads", predict_len=200, temperature=0.8))

In [ ]:
print(generate(model, start_str="Two roads", predict_len=100, temperature=0.3))

In [ ]:
print(generate(model, start_str="learning", predict_len=20, temperature=0.8))